# Combine nmaps

#### In this notebook, the goal is to import the neuromaps data for glu-related maps and generate some binarized masks.

1. Import packages and maps
2. Combine receptor maps of interest

## Load Packages

In [1]:
import pandas as pd
import nilearn
from nilearn.datasets import fetch_atlas_schaefer_2018
from neuromaps.datasets import fetch_atlas
from nilearn.image import binarize_img
from nilearn import image as nimage
from neuromaps.images import (load_nifti, load_data)
from neuromaps.parcellate import Parcellater
import warnings
from neuromaps.datasets import fetch_fslr, fetch_annotation, available_annotations
from neuromaps import datasets, images, nulls, resampling, stats, transforms, parcellate
from neuromaps.parcellate import Parcellater 
from neuromaps.resampling import resample_images
from neuromaps.images import dlabel_to_gifti
import neuromaps
import nibabel as nib
import os
import numpy as np
import mayavi
from PIL import Image
import hcp_utils as hcp
import nilearn.plotting as plotting
import warnings
import nibabel as nib
import pylab as plt
from nilearn.image import resample_img
from scipy.stats import zscore
#import abagen ## This is important for certification purposes 
from netneurotools import datasets as nntdata
warnings.filterwarnings('ignore')

pixdim[1,2,3] should be non-zero; setting 0 dims to 1


## Import Data and Set Variables

In [1]:
# List annotations of interest. 
ANNOTATIONS = [
    ('dukart2018', 'flumazenil', 'MNI152', '3mm'),   # GABA
    ('rosaneto', 'abp688', 'MNI152', '1mm'),         # mGluR1
    ('dubois2015', 'abp688', 'MNI152', '1mm'),       # mGluR2
    ('smart2019', 'abp688', 'MNI152', '1mm'),        # mGluR3
 ]


                 path+'5HT1a_way_hc36_savli.nii',
                 #path+'5HT1a_cumi_hc8_beliveau.nii',
                 path+'5HT1b_az_hc36_beliveau.nii',
                 path+'5HT1b_p943_hc22_savli.nii',
                 path+'5HT1b_p943_hc65_gallezot.nii.gz',
                 path+'5HT2a_cimbi_hc29_beliveau.nii',
                 path+'5HT2a_alt_hc19_savli.nii',
                 path+'5HT2a_mdl_hc3_talbot.nii.gz',
                 path+'5HT4_sb20_hc59_beliveau.nii',
                 path+'5HT6_gsk_hc30_radhakrishnan.nii.gz',
                 path+'5HTT_dasb_hc100_beliveau.nii',
                 path+'5HTT_dasb_hc30_savli.nii',
                 path+'A4B2_flubatine_hc30_hillmer.nii.gz',
                 path+'CB1_omar_hc77_normandin.nii.gz',
                 path+'CB1_FMPEPd2_hc22_laurikainen.nii',
                 path+'D1_SCH23390_hc13_kaller.nii',
                 path+'D2_fallypride_hc49_jaworska.nii',
                 path+'D2_flb457_hc37_smith.nii.gz',
                 path+'D2_flb457_hc55_sandiego.nii.gz',
                 path+'D2_raclopride_hc7_alakurtti.nii',
                 path+'DAT_fpcit_hc174_dukart_spect.nii',
                 path+'DAT_fepe2i_hc6_sasaki.nii.gz',
                 path+'GABAa-bz_flumazenil_hc16_norgaard.nii',
                 path+'GABAa_flumazenil_hc6_dukart.nii',
                 path+'H3_cban_hc8_gallezot.nii.gz',
                 path+'M1_lsn_hc24_naganawa.nii.gz',
                 path+'mGluR5_abp_hc22_rosaneto.nii',
                 path+'mGluR5_abp_hc28_dubois.nii',
                 path+'mGluR5_abp_hc73_smart.nii',
                 path+'MU_carfentanil_hc204_kantonen.nii',
                 path+'MU_carfentanil_hc39_turtonen.nii',
                 path+'NAT_MRB_hc77_ding.nii.gz',
                 path+'NAT_MRB_hc10_hesse.nii',
                 path+'NMDA_ge179_hc29_galovic.nii.gz',
                 #path+'VAChT_feobv_hc3_spreng.nii',
                 path+'VAChT_feobv_hc4_tuominen.nii',
                 path+'VAChT_feobv_hc5_bedard_sum.nii',
                 path+'VAChT_feobv_hc18_aghourian_sum.nii']



# Unmasked
('gallezot2010', 'p943', 'MNI152', '1mm') #5HT1b_p943_hc65_gallezot
## Not in annotations list????  # 5HT2a_mdl_hc3_talbot
('radnakrishnan2018', 'gsk215083', 'MNI152', '1mm') #5HT6_gsk_hc30_radhakrishnan
('hillmer2016', 'flubatine', 'MNI152', '1mm') #A4B2_flubatine_hc30_hillmer
('laurikainen2018', 'fmpepd2', 'MNI152', '1mm') #CB1_FMPEPd2_hc22_laurikainen
('normandin2015', 'omar', 'MNI152', '1mm') #CB1_omar_hc77_normandin
('jaworska2020', 'fallypride', 'MNI152', '1mm') # D2_fallypride_hc49_jaworska
('sandiego2015', 'flb457', 'MNI152', '1mm') # D2_flb457_hc55_sandiego
('smith2017', 'flb457', 'MNI152', '1mm') # D2_flb457_hc37_smith
('alarkurtti2015', 'raclopride', 'MNI152', '3mm') # D2_raclopride_hc7_alakurtti
('sasaki2012', 'fepe2i', 'MNI152', '1mm') # DAT_fepe2i_hc6_sasaki
## Not in annotations list????  # DAT_fpcit_hc174_dukart_spect
## Not in annotations list????  # FDOPA_fluorodopa_hc12_gomez
('gallezot2017', 'gsk189254', 'MNI152', '1mm') # H3_cban_hc8_gallezot
('naganawa2020', 'lsn3172176', 'MNI152', '1mm')  # M1_lsn_hc24_naganawa
('turtonen2020', 'carfentanil', 'MNI152', '1mm') # MU_carfentanil_hc39_turtonen
('ding2010', 'mrb', 'MNI152', '1mm') #NAT_MRB_hc77_ding
('tuominen', 'feobv', 'MNI152', '2mm') # VAChT_feobv_hc4_tuominen
('bedard2019', 'feobv', 'MNI152', '1mm') #VAChT_feobv_hc5_bedard_sum
## Not in annotations list????  # VAChT_feobv_hc18_aghourian_sum

# No white matter:
# ('beliveau2017', 'cumi101', 'MNI152', '1mm') 
# ('beliveau2017', 'az10419369', 'MNI152', '1mm')
# ('beliveau2017', 'cimbi36', 'MNI152', '1mm')
# ('beliveau2017', 'sb207145', 'MNI152', '1mm')
# ('beliveau2017', 'dasb', 'MNI152', '1mm')
# ('fazio2016', 'madam', 'MNI152', '3mm')
# ('savli2012', 'way100635', 'MNI152', '3mm')
# ('savli2012', 'p943', 'MNI152', '3mm')
# ('savli2012', 'altanserin', 'MNI152', '3mm') # 5HT2a_alt_hc19_savli
# ('kaller2017', 'sch23390', 'MNI152', '3mm')
# ('kantonen2020', 'carfentanil', 'MNI152', '3mm') # i think no wm?
# ('hesse2017', 'methylreboxetine', 'MNI152', '3mm')
# ('aghourian2017', 'feobv', 'MNI152', '1mm')

# https://ftp.bigbrainproject.org/bigbrain-ftp/BigBrainRelease.2015/Layer_Segmentation/3D_Surfaces/PLoSBiology2020/
              
# Generate an empty list to add transformed maps
nmaps_transformed = []
nmaps_transformed_array = []
# Set paths
outpath = "/Users/pecsok/projects/Neuromaps/pecsok_pfns/neuromaps/data/maps/PET_nifti_images/"
nmda_path = "/Users/pecsok/projects/Neuromaps/pecsok_pfns/neuromaps/data/maps/PET_nifti_images/"

# Notes:
#  1. NMDA is not in available annotations list but was copied from Hansen git repo
#       Already in MNI 152 at 1mm resolution. transfer to outt
#  2. We are excluding GABA map ('norgaard2021', 'flumazenil', 'MNI152', '1mm') 
#      Only has cortical data. Consider re-adding for later analyses or future studies. 

In [8]:
# Set path for downloading neuromaps data
data_dir = '/Users/pecsok/projects/Neuromaps'

### Load in neuromaps of interest

In [9]:
# 6th gen MNI template:
mni152 = fetch_atlas('MNI152', '1mm')
mninifti = nib.load(mni152['6Asym_T1w'])
us_mninifti = resample_img(mninifti, target_affine=np.eye(3)*0.82, interpolation='nearest')

# 6th gen MNI mask
mnimask = nib.load(mni152['6Asym_brainmask'])
us_mnimask = resample_img(mnimask, target_affine=np.eye(3)*0.82, interpolation='nearest')
maskarray = us_mnimask.get_fdata()

# Fetch annotations
imgs=[]
for (src, desc, space, den) in ANNOTATIONS:
    img = datasets.fetch_annotation(source=src, desc=desc,
                                    space=space, den=den)
    imgs.append(img)


 ...done. (2 seconds, 0 min)


In [5]:
# Resample GABA
resampled_GABA=neuromaps.resampling.resample_images(src=imgs[0],src_space='MNI152',trg_space='MNI152',
                                                    trg=us_mninifti,method='linear',resampling='transform_to_trg')
# Skull Strip
print(resampled_GABA)
array = resampled_GABA[0].get_fdata()
masked_array = array * maskarray
nib.save(nib.Nifti1Image(masked_array,resampled_GABA[0].affine),outpath +'resampled_GABA_masked.nii.gz')

# Z Score
masked_values = masked_array[masked_array > 0]
zscored_values = zscore(masked_values)
zmasked_array = np.zeros_like(masked_array)
zmasked_array[masked_array > 0] = zscored_values


nib.save(nib.Nifti1Image(zmasked_array,resampled_GABA[0].affine),outpath +'resampled_GABA_zmasked.nii.gz')

(<nibabel.nifti1.Nifti1Image object at 0x30c84ef50>, <nibabel.nifti1.Nifti1Image object at 0x34fe1d8d0>)


In [11]:
#### Resample mGluR maps
resampled_mglu1=neuromaps.resampling.resample_images(src=imgs[1],src_space='MNI152',trg_space='MNI152',
                                                        trg=us_mninifti,method='linear',resampling='transform_to_trg')
resampled_mglu2=neuromaps.resampling.resample_images(src=imgs[2],src_space='MNI152',trg_space='MNI152',
                                                        trg=us_mninifti,method='linear',resampling='transform_to_trg')
resampled_mglu3=neuromaps.resampling.resample_images(src=imgs[3],src_space='MNI152',trg_space='MNI152',
                                                        trg=us_mninifti,method='linear',resampling='transform_to_trg')
# Save individual maps
nib.save(nib.Nifti1Image(resampled_mglu1[0].get_fdata(),resampled_mglu1[0].affine),outpath +'resampled_mGluR1.nii.gz')
nib.save(nib.Nifti1Image(resampled_mglu2[0].get_fdata(),resampled_mglu2[0].affine),outpath +'resampled_mGluR2.nii.gz')
nib.save(nib.Nifti1Image(resampled_mglu3[0].get_fdata(),resampled_mglu3[0].affine),outpath +'resampled_mGluR3.nii.gz')

# Combine maps 
combined_mglu_array = (22*resampled_mglu1[0].get_fdata()+28*resampled_mglu2[0].get_fdata()+73*resampled_mglu3[0].get_fdata())/(22+28+73)
#nib.save(nib.Nifti1Image(combined_mglu_array,resampled_mglu1[0].affine),outpath +'resampled_combined_mGluR5.nii.gz')

# Skull Strip
masked_array = combined_mglu_array * maskarray
nib.save(nib.Nifti1Image(masked_array,resampled_GABA[0].affine),outpath +'resampled_mGluR5_masked.nii.gz')

# Z Score
masked_values = masked_array[masked_array > 0]
zscored_values = zscore(masked_values)
zmasked_array = np.zeros_like(masked_array)
zmasked_array[masked_array > 0] = zscored_values

nib.save(nib.Nifti1Image(zmasked_array,resampled_nmda[0].affine),outpath +'resampled_mGluR5_zmasked.nii.gz')

In [10]:
# Resample NMDA 
nmda = nib.load(nmda_path+'NMDA_ge179_hc29_galovic.nii.gz')
resampled_nmda=neuromaps.resampling.resample_images(src=nmda,src_space='MNI152',trg_space='MNI152',
                                                        trg=us_mninifti,method='linear',resampling='transform_to_trg')
# Skull Strip
array = resampled_nmda[0].get_fdata()
masked_array = array * maskarray
nib.save(nib.Nifti1Image(masked_array,resampled_nmda[0].affine),outpath +'resampled_NMDA_masked.nii.gz')

# Z Score
masked_values = masked_array[masked_array > 0]
zscored_values = zscore(masked_values)
zmasked_array = np.zeros_like(masked_array)
zmasked_array[masked_array > 0] = zscored_values

nib.save(nib.Nifti1Image(zmasked_array,resampled_nmda[0].affine),outpath +'resampled_NMDA_zmasked.nii.gz')

In [12]:
#### Resample D2 maps
resampled_D2_1=neuromaps.resampling.resample_images(src=imgs[4],src_space='MNI152',trg_space='MNI152',
                                                        trg=us_mninifti,method='linear',resampling='transform_to_trg')
resampled_D2_2=neuromaps.resampling.resample_images(src=imgs[5],src_space='MNI152',trg_space='MNI152',
                                                        trg=us_mninifti,method='linear',resampling='transform_to_trg')

# Save individual maps
nib.save(nib.Nifti1Image(resampled_D2_1[0].get_fdata(),resampled_D2_1[0].affine),outpath +'resampled_D2_1.nii.gz')
nib.save(nib.Nifti1Image(resampled_D2_2[0].get_fdata(),resampled_D2_2[0].affine),outpath +'resampled_D2_2.nii.gz')

# Combine maps 
combined_D2_array = (55*resampled_D2_1[0].get_fdata()+37*resampled_D2_2[0].get_fdata())/(55+37)
#nib.save(nib.Nifti1Image(combined_mglu_array,resampled_mglu1[0].affine),outpath +'resampled_combined_mGluR5.nii.gz')

# Skull Strip
masked_array = combined_D2_array * maskarray
nib.save(nib.Nifti1Image(masked_array,resampled_nmda[0].affine),outpath +'resampled_D2_masked.nii.gz')

# Z Score
masked_values = masked_array[masked_array > 0]
zscored_values = zscore(masked_values)
zmasked_array = np.zeros_like(masked_array)
zmasked_array[masked_array > 0] = zscored_values

nib.save(nib.Nifti1Image(zmasked_array,resampled_nmda[0].affine),outpath +'resampled_D2_zmasked.nii.gz')

In [13]:
# Resample additional volumetric maps of interest.

mapnames = ['sv2a','cogpc1','cbf']
indices = [6,7,8]

for i in range(len(mapnames)):
    name=mapnames[i]
    index=indices[i]
    print(imgs[index])
    resampled=neuromaps.resampling.resample_images(src=imgs[6],src_space='MNI152',trg_space='MNI152',
                                                        trg=us_mninifti,method='linear',resampling='transform_to_trg')
    array = resampled[0].get_fdata()
    masked_array = array * maskarray
    nib.save(nib.Nifti1Image(masked_array,resampled[0].affine),outpath +'resampled_'+name+'_masked.nii.gz')
    
    # Z Score
    masked_values = masked_array[masked_array > 0]
    zscored_values = zscore(masked_values)
    zmasked_array = np.zeros_like(masked_array)
    zmasked_array[masked_array > 0] = zscored_values    
    nib.save(nib.Nifti1Image(zmasked_array,resampled[0].affine),outpath +'resampled_'+name+'_zmasked.nii.gz')


/Users/pecsok/neuromaps-data/annotations/finnema2016/ucbj/MNI152/source-finnema2016_desc-ucbj_space-MNI152_res-1mm_feature.nii.gz
/Users/pecsok/neuromaps-data/annotations/neurosynth/cogpc1/MNI152/source-neurosynth_desc-cogpc1_space-MNI152_res-2mm_feature.nii.gz
/Users/pecsok/neuromaps-data/annotations/satterthwaite2014/meancbf/MNI152/source-satterthwaite2014_desc-meancbf_space-MNI152_res-1mm_feature.nii.gz
